# Skin Disease Detection System (ResNet18, PyTorch)
Educational demo on a **synthetic dataset**. Not a medical tool.

Run the cells from top to bottom.

In [ ]:
# STEP 0: Install (run once)
!pip install torch torchvision

In [ ]:
# STEP 1: IMPORT LIBRARIES
import os
import numpy as np
from PIL import Image
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [ ]:
# STEP 2: DEVICE SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# STEP 3: CREATE SYNTHETIC DATASET
base_path = "/content/dataset"
classes = ["melanoma", "nevus", "eczema", "psoriasis"]

for split in ["train", "val"]:
    for cls in classes:
        os.makedirs(f"{base_path}/{split}/{cls}", exist_ok=True)

def generate_image(label):
    img = np.zeros((224, 224, 3), dtype=np.uint8)

    if label == "melanoma":
        img[:] = [120, 50, 50]
    elif label == "nevus":
        img[:] = [180, 120, 90]
    elif label == "eczema":
        img[:] = [200, 120, 120]
    elif label == "psoriasis":
        img[:] = [230, 230, 230]

    noise = np.random.randint(0, 50, (224, 224, 3))
    img = np.clip(img + noise, 0, 255)
    return Image.fromarray(img.astype("uint8"))

for cls in classes:
    for i in range(50):
        generate_image(cls).save(f"{base_path}/train/{cls}/{cls}_{i}.jpg")
    for i in range(10):
        generate_image(cls).save(f"{base_path}/val/{cls}/{cls}_{i}.jpg")

print("✅ Dataset Created")

In [ ]:
# STEP 4: LOAD DATASET
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.ImageFolder("/content/dataset/train", transform=transform)
val_data = datasets.ImageFolder("/content/dataset/val", transform=transform)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)

class_names = train_data.classes
print("Classes:", class_names)

In [ ]:
# STEP 5: LOAD MODEL (UPDATED PYTORCH)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)
print("✅ Model Ready")

In [ ]:
# STEP 6: LOSS & OPTIMIZER
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# STEP 7: TRAINING
def train_model(epochs=10):
    print("🚀 Training Started")
    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} → Accuracy: {acc:.4f}")
    print("✅ Training Completed")

train_model()

In [ ]:
# STEP 8: EVALUATION
def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print("Validation Accuracy:", correct / total)

evaluate()

In [ ]:
# STEP 9: SAVE MODEL
torch.save(model.state_dict(), "/content/skin_model.pth")
print("✅ Model Saved")

In [ ]:
# STEP 10: PREDICTION FUNCTION
def predict_image(path):
    model.eval()
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    return class_names[pred.item()], conf.item()

In [ ]:
# STEP 11: TEST PREDICTION
test_img = "/content/dataset/val/melanoma/melanoma_1.jpg"
label, conf = predict_image(test_img)

print("Prediction:", label)
print("Confidence:", conf)

# STEP 12: RISK WARNING SYSTEM
if conf < 0.6:
    print("⚠️ Low Confidence – Consult Doctor")
elif conf < 0.8:
    print("⚠️ Medium Risk")
else:
    print("✅ High Confidence Prediction")

In [ ]:
# 10.1 Scatter Plot for Prediction Confidence
import matplotlib.pyplot as plt

classes = ["melanoma", "nevus", "eczema", "psoriasis"]
confidence = [0.95, 0.85, 0.78, 0.92]
colors = ["red", "brown", "pink", "gray"]
x = range(len(classes))
plt.scatter(x, confidence, c=colors, s=100)
plt.xticks(x, classes)
plt.xlabel("Disease Type")
plt.ylabel("Confidence Score")
plt.title("Colored Scatter Plot of Predictions")
plt.show()